<a href="https://colab.research.google.com/github/priyankabolem/PlantDiseasesDetection/blob/main/PlantDiseaseDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 1: Download Dataset from Kaggle

In [33]:
from google.colab import files
files.upload()  # Select kaggle.json when prompted

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"priyabolem","key":"46cc8eea19d145e50ecdefb3827edeb9"}'}

Step 2: Install Kaggle CLI & Download Dataset

In [3]:
!pip install -q kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d vipoooool/new-plant-diseases-dataset

# Extract dataset
import zipfile
with zipfile.ZipFile("new-plant-diseases-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("plant_disease_dataset")

print("Dataset extracted successfully!")

Dataset URL: https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
License(s): copyright-authors
100% 2.70G/2.70G [02:02<00:00, 24.2MB/s]
100% 2.70G/2.70G [02:02<00:00, 23.6MB/s]
Dataset extracted successfully!


Step 3: Load and Preprocess Dataset

In [17]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define Correct Paths
dataset_root = "/content/plant_disease_dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
train_dir = os.path.join(dataset_root, "train")
valid_dir = os.path.join(dataset_root, "valid")

# Check if paths exist
if not os.path.exists(train_dir):
    print(f"❌ Train directory NOT found: {train_dir}")
if not os.path.exists(valid_dir):
    print(f"❌ Validation directory NOT found: {valid_dir}")

# Image Data Preprocessing
img_size = (150, 150)
batch_size = 32

train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)
train_generator = train_datagen.flow_from_directory(
    train_dir,  # Updated path
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

validation_generator = train_datagen.flow_from_directory(
    valid_dir,  # Updated path
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

print("✅ Dataset loaded successfully!")

Found 70295 images belonging to 38 classes.
Found 17572 images belonging to 38 classes.
✅ Dataset loaded successfully!


Step 4: Build and Train CNN Model

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build CNN Model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(len(train_generator.class_indices), activation='softmax')
])

# Compile Model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train Model
epochs = 10
model.fit(train_generator, validation_data=validation_generator, epochs=epochs)

# Save Model
model.save("plant_disease_model.h5")
print("Model training completed and saved successfully!")

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 150s 66ms/step - accuracy: 0.3248 - loss: 2.4465 - val_accuracy: 0.7892 - val_loss: 0.7225
Epoch 2/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 103s 47ms/step - accuracy: 0.6621 - loss: 1.0782 - val_accuracy: 0.8538 - val_loss: 0.4857
Epoch 3/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 102s 47ms/step - accuracy: 0.7442 - loss: 0.7968 - val_accuracy: 0.8639 - val_loss: 0.4287
Epoch 4/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 100s 46ms/step - accuracy: 0.7912 - loss: 0.6376 - val_accuracy: 0.8931 - val_loss: 0.3419
Epoch 5/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 102s 47ms/step - accuracy: 0.8278 - loss: 0.5174 - val_accuracy: 0.8938 - val_loss: 0.3376
Epoch 6/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 102s 46ms/step - accuracy: 0.8492 - loss: 0.4555 - val_accuracy: 0.8982 - val_loss: 0.3272
Epoch 7/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 108s 49ms/step - accuracy: 0.8676 - loss: 0.3943 - val_accuracy: 0.9131 - val_loss: 0.2836
Epoch 8/10
2197/2197 ━━━━━━━━━━━━━━━━━━━━ 99s 45ms/step - accuracy: 0

Model training completed and saved successfully!


Step 5: Test the Model on a Sample Image

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
import os
print(os.listdir())  # Lists all files and folders in your working directory

['.config', 'kaggle.json', 'plant_disease_model.h5', 'plant_disease_dataset', '.ipynb_checkpoints', 'new-plant-diseases-dataset.zip', 'sample_data']


In [26]:
import os
print(os.listdir())  # Lists all files in the current directory

['.config', 'kaggle.json', 'plant_disease_model.h5', 'plant_disease_dataset', '.ipynb_checkpoints', 'new-plant-diseases-dataset.zip', 'sample_data']


In [27]:
model.save("plant_disease_model.keras")  # Saves in the new Keras format

In [28]:
print(os.listdir())  # Check if the model file appears

['.config', 'kaggle.json', 'plant_disease_model.h5', 'plant_disease_dataset', '.ipynb_checkpoints', 'new-plant-diseases-dataset.zip', 'plant_disease_model.keras', 'sample_data']


In [9]:
import os
print(os.listdir("/content/drive/MyDrive"))  # Lists files in your Google Drive root folder

['HP_Results.zip', 'BOLEM723.jpg', 'bolem333.jpg', 'SearchTIN.txt', 'Performance Based Seismic Evaluation of Industrial Chimneys _ Project.ppt', 'Performance Based Seismic Evaluation of Industrial Chimneys _ Project.ppt.gslides', 'IMG_7596.JPG', 'IMG_7597.JPG', 'IMG_7594.JPG', 'IMG_7598.JPG', 'Pamarru-4  BIOMETRIC FAILURE  PENSIONERS LIST(1).xlsx', 'Krishna PDO payments.xlsx', 'SECRETARIAT (2).xls', 'Krishna PDO payments-1.xlsx', 'Pamarru-4 biometric failure.xlsx', '5_6087145024204571249.xlsx', '9004EC45-34D8-49DA-B0F2-A380311489B3.jpeg', '5_6201764490015409048.xlsx', 'Pamarru-4 Spandana Login Details.xlsx', 'Pamarru-4 volunteers mapping to functionaries .xlsx', 'Pamarru-4 volunteers mapping to functionaries .gsheet', 'Pamarru-4.xlsx', 'Pamarru-4 sachivalayam covid samples (1).gsheet', 'Pamarru-4 sachivalayam covid samples.xlsx', 'Pamarru-4 sachivalayam covid samples.gsheet', 'Pamarru-4 ATTENDANCE (1).gsheet', 'Pamarru-4 ATTENDANCE.xlsx', 'Pamarru-4 ATTENDANCE.gsheet', 'Pamarru-4 trans

In [11]:
print(os.path.exists(sample_image_path))  # Should return True

True


In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array
import numpy as np
import os

# ✅ Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ✅ Step 2: Define Correct File Paths
model_path = "/content/drive/MyDrive/plant_disease_model.keras"  # Update if saved elsewhere
sample_image_path = "/content/drive/MyDrive/Tomatodiseased.jpg"  # Corrected image path

# ✅ Step 3: Ensure the Model Exists
if os.path.exists(model_path):
    model = tf.keras.models.load_model(model_path)
    print("✅ Model loaded successfully!")
else:
    raise FileNotFoundError(f"❌ Model file '{model_path}' not found. Check the path.")

# ✅ Step 4: Ensure the Image File Exists
if not os.path.exists(sample_image_path):
    raise FileNotFoundError(f"❌ Image file '{sample_image_path}' not found. Check the path.")

# ✅ Step 5: Load and Preprocess the Image
img = load_img(sample_image_path, target_size=(150, 150))  # Resize image to model input size
img_array = img_to_array(img) / 255.0  # Normalize image
img_array = np.expand_dims(img_array, axis=0)  # Add batch dimension

# ✅ Step 6: Make Predictions
predictions = model.predict(img_array)

# ✅ Step 7: Decode Predictions
class_indices = {0: 'Healthy', 1: 'Diseased'}  # Replace with actual class labels
predicted_class = np.argmax(predictions, axis=1)[0]
predicted_label = class_indices.get(predicted_class, "Unknown")

# ✅ Step 8: Print the Prediction Result
print(f"🌿 Predicted class: {predicted_label}")

Mounted at /content/drive


FileNotFoundError: ❌ Model file '/content/drive/MyDrive/plant_disease_model.keras' not found. Check the path.

Step 6: Deploy Model with Streamlit

In [ ]:
from google.colab import files
files.download("plant_disease_model.h5")